# 05A Chat with your project code

**Problem:** A student club has inherited a small registration project. Find the code that controls seats, explain it using file and line references, then use an available coding assistant to add one small feature and verify it.

This is a lightweight classroom adaptation inspired by [Chat with Code](https://github.com/patchy631/ai-engineering-hub/tree/042aeb5067384981eeba6a07fa882f436bb1660d/chat-with-code). We use original workshop code and Python's standard library. The original project uses LlamaIndex, Milvus, Cleanlab and a hosted model; those services are not required here.

**40-minute core:** 5 minutes run the project; 8 retrieve code; 7 prepare a grounded prompt; 15 implement and test; 5 explain.

**Important:** The search step returns source snippets. It is labelled RETRIEVAL_ONLY and does not claim to be an LLM answer. Use your existing assistant manually for the coding request; this notebook never sends your files or credentials to an external service.


## 1 Prepare a fresh workspace

**What this cell does:** Loads standard Python modules and creates a temporary workshop folder.

**Run:** Use a CPU runtime. No package installation or API key is needed.

**Expected:** LOCAL_PYTHON and a workspace folder name. Download your work before the runtime ends.


In [ ]:
from pathlib import Path
import ast, copy, difflib, hashlib, importlib.util, io, json, math, re, sys, tempfile, unittest, zipfile
from collections import Counter

# A new folder avoids overwriting an unrelated student project.
WORKSPACE = Path(tempfile.mkdtemp(prefix="bgs05a_"))
MODE = "LOCAL_PYTHON"
print("MODE:", MODE, "| Workspace:", WORKSPACE.name)


## 2 Create the three-file project

**What this cell does:** Writes a complete, working event-registration example: bookings.py, app.py and README.md.

**Explain:** State changes belong in EventBooking. Read the conditions before the list is changed.

**Expected:** Three files. Running this cell again restores the supplied baseline in this temporary workspace, so preserve your edited source before rerunning it.


In [ ]:
BASELINE_BOOKINGS_SOURCE = '"""Fictional event registration; no real college records are changed."""\n\nclass EventBooking:\n    def __init__(self, capacity=2):\n        if type(capacity) is not int or capacity < 1:\n            raise ValueError("Capacity must be a positive integer")\n        self.capacity = capacity\n        self.registered = []\n\n    def remaining(self):\n        return self.capacity - len(self.registered)\n\n    def register(self, student_id):\n        # Reject malformed IDs before changing the registration list.\n        if not isinstance(student_id, str) or len(student_id) != 4 or not student_id.startswith("S") or not student_id[1:].isdigit():\n            return {"ok": False, "reason": "invalid_id", "remaining": self.remaining()}\n        # Duplicate registration must not consume another seat.\n        if student_id in self.registered:\n            return {"ok": False, "reason": "duplicate", "remaining": self.remaining()}\n        if self.remaining() == 0:\n            return {"ok": False, "reason": "full", "remaining": 0}\n        self.registered.append(student_id)\n        return {"ok": True, "remaining": self.remaining()}\n\n    def cancel(self, student_id):\n        if student_id not in self.registered:\n            return {"ok": False, "reason": "not_registered", "remaining": self.remaining()}\n        self.registered.remove(student_id)\n        return {"ok": True, "remaining": self.remaining()}\n'
APP_SOURCE = '"""A small entry point that uses EventBooking."""\nfrom bookings import EventBooking\n\ndef run_demo():\n    booking = EventBooking(capacity=2)\n    return [booking.register("S001"), booking.register("S002"),\n            booking.register("S003"), booking.cancel("S001"),\n            booking.register("S003")]\n'
PROJECT_README = '# CodeSprint registration demo\n\nThis practice project manages seats for one fictional event.\nbookings.py owns validation, registration and cancellation.\napp.py demonstrates two seats, a full event and a cancellation.\nThe workshop notebook contains the tests.\nIDs use S followed by three digits, such as S001.\nNo real registration, login, payment or network call occurs.\n'
PROJECT_FILES = {"bookings.py": BASELINE_BOOKINGS_SOURCE, "app.py": APP_SOURCE, "README.md": PROJECT_README}
for relative, text in PROJECT_FILES.items():
    # Only the three supplied filenames are written inside our fresh workspace.
    (WORKSPACE / relative).write_text(text, encoding="utf-8")
print("Project files:", ", ".join(PROJECT_FILES))


## 3 Index source blocks with their original lines

**What this cell does:** Uses Python's syntax tree to locate functions and methods. It stores each snippet with its filename and starting/ending line.

**Explain:** File references let us verify an explanation. The index reads only our three supplied project files.

**Expected:** A small source index. Rebuild it after changing code so references stay current.


In [ ]:
def build_code_index():
    blocks = []
    for relative in ["bookings.py", "app.py"]:
        lines = (WORKSPACE / relative).read_text(encoding="utf-8").splitlines()
        tree = ast.parse("\n".join(lines))
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                blocks.append({"file": relative, "name": node.name,
                               "start": node.lineno, "end": node.end_lineno,
                               "text": "\n".join(lines[node.lineno-1:node.end_lineno])})
    return blocks

# Use lexical relevance for this tiny project; no embedding model is called.
STOP_WORDS = {"a", "an", "the", "is", "are", "to", "do", "does", "where", "how", "with", "of", "in", "for", "this"}
def terms(text):
    return [word for word in re.findall(r"[a-z_]+", text.lower()) if word not in STOP_WORDS]

def search_code(question, top_k=3):
    if not isinstance(question, str) or not question.strip():
        return []
    blocks = build_code_index()
    query = set(terms(question))
    frequency = Counter(term for block in blocks for term in set(terms(block["text"])))
    scored = []
    for block in blocks:
        words = Counter(terms(block["text"]))
        # Rare matching terms receive more weight than words present everywhere.
        score = sum((1 + math.log((1 + len(blocks)) / (1 + frequency[word]))) * min(words[word], 3)
                    for word in query if words[word])
        if score > 0:
            scored.append({**block, "score": round(score, 3)})
    return sorted(scored, key=lambda row: (-row["score"], row["file"], row["start"]))[:top_k]

print("Indexed source blocks:", len(build_code_index()))


## 4 Ask a code question and inspect the evidence

**What this cell does:** Retrieves the most relevant code blocks and prints their exact source lines.

**Student task 1:** Replace QUESTION with a question about cancellation or capacity. Open the cited lines and check whether they support your explanation.

**Expected:** RETRIEVAL_ONLY followed by file:line references. No matching source produces an honest empty result.


In [ ]:
QUESTION = "Where does the code reject duplicate registrations?"
retrieved = search_code(QUESTION)
print("RETRIEVAL_ONLY — source snippets, not a generated answer")
if not retrieved:
    print("No matching source found. Try names or terms from the project.")
for block in retrieved:
    print(f"\n[{block['file']}:{block['start']}-{block['end']}] {block['name']} score={block['score']}")
    for number, line in enumerate(block["text"].splitlines(), block["start"]):
        print(f"{number:>3}: {line}")


## 5 Build a grounded assistant request

**What this cell does:** Creates a prompt containing the current source and a precise feature requirement.

**Student task 2:** Paste the prompt into an assistant you can access. Ask it to explain the relevant lines and propose a small patch. You can also implement the change yourself and record that route.

**Expected:** A copyable prompt. This step makes no model request. Read any suggested code before applying it.


In [ ]:
FEATURE_REQUEST = "Add a status() method returning capacity, registered_count and remaining. It must not change registrations."
source_packet = "\n\n".join(f"FILE: {name}\n{(WORKSPACE / name).read_text(encoding='utf-8')}" for name in PROJECT_FILES)
coding_prompt = f"""You are helping with a small, fictional college registration project.
Explain the relevant file and lines, then propose a minimal patch.
Requirement: {FEATURE_REQUEST}
Preserve input validation, existing behavior and all baseline tests. Add no dependencies.
Return a short plan, the proposed code change and tests; do not claim tests ran.
Current project source:
{source_packet}
"""
print(coding_prompt)


## 6 Run the unchanged baseline tests

**What this cell does:** Loads the supplied local module and checks eight behaviours, including duplicate rejection, full capacity and cancellation.

**Explain:** These tests check Python state, not an assistant's confidence. Keep their expected values when adding the new feature.

**Expected:** 8 / 8 checks pass on the supplied baseline. The runner reports failures as results rather than an unhandled notebook exception.


In [ ]:
def load_booking_class():
    # Load only the explicit project file the student is working on.
    spec = importlib.util.spec_from_file_location("bgs05a_bookings", WORKSPACE / "bookings.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.EventBooking

def baseline_tests():
    Booking = load_booking_class()
    class BookingChecks(unittest.TestCase):
        def test_initial_capacity(self):
            self.assertEqual(Booking(2).remaining(), 2)
        def test_success_consumes_one_seat(self):
            booking = Booking(2)
            self.assertTrue(booking.register("S001")["ok"])
            self.assertEqual(booking.remaining(), 1)
        def test_duplicate_keeps_state(self):
            booking = Booking(2); booking.register("S001")
            self.assertEqual(booking.register("S001")["reason"], "duplicate")
            self.assertEqual(booking.registered, ["S001"])
        def test_full_event_rejects(self):
            booking = Booking(1); booking.register("S001")
            self.assertEqual(booking.register("S002")["reason"], "full")
            self.assertEqual(booking.remaining(), 0)
        def test_cancel_restores_seat(self):
            booking = Booking(2); booking.register("S001"); booking.cancel("S001")
            self.assertEqual(booking.remaining(), 2)
        def test_unknown_cancellation_keeps_state(self):
            booking = Booking(2)
            self.assertFalse(booking.cancel("S099")["ok"])
            self.assertEqual(booking.remaining(), 2)
        def test_invalid_id_keeps_state(self):
            for value in ["", "S01", "X001", None, 12]:
                booking = Booking(2)
                self.assertEqual(booking.register(value)["reason"], "invalid_id")
                self.assertEqual(booking.registered, [])
        def test_invalid_capacity(self):
            for value in [0, -1, 1.5, True]:
                with self.assertRaises(ValueError): Booking(value)
    stream = io.StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(BookingChecks))
    summary = {"passed": result.testsRun-len(result.failures)-len(result.errors),
               "total": result.testsRun, "ok": result.wasSuccessful()}
    print("Baseline checks:", summary)
    if not result.wasSuccessful():
        print("Review the project and failing checks:\n" + stream.getvalue())
    return summary

before_checks = baseline_tests()


## 7 Review and apply your small change

**What this cell does:** Shows a diff between the current file and your reviewed replacement. It writes the replacement only when APPLY_REVIEWED_CHANGE is True.

Replace REVIEWED_BOOKINGS_SOURCE with your updated module, inspect the diff and then set the flag. The default source is unchanged, so Run all leaves a working baseline.

Run code only after you understand it. This flag records a human workflow step; it is not a security sandbox.


In [ ]:
# Replace this value with your reviewed complete bookings.py source.
REVIEWED_BOOKINGS_SOURCE = BASELINE_BOOKINGS_SOURCE
APPLY_REVIEWED_CHANGE = False
current_source = (WORKSPACE / "bookings.py").read_text(encoding="utf-8")
print("".join(difflib.unified_diff(current_source.splitlines(True), REVIEWED_BOOKINGS_SOURCE.splitlines(True),
                                 fromfile="current/bookings.py", tofile="reviewed/bookings.py")) or "No source difference yet.")
if APPLY_REVIEWED_CHANGE:
    # Syntax validation does not establish safety or correctness; review comes first.
    ast.parse(REVIEWED_BOOKINGS_SOURCE)
    (WORKSPACE / "bookings.py").write_text(REVIEWED_BOOKINGS_SOURCE, encoding="utf-8")
    print("Reviewed source saved. Run the next checks.")
else:
    print("Change not applied. Review the code and enable the flag when ready.")


## 8 Verify the same behaviours and your new feature

**What this cell does:** Repeats the unchanged baseline checks, then optionally verifies status().

**Student task 3:** Add the status() feature, enable RUN_FEATURE_CHECK, and show that calling it does not mutate the registration list. Add one further test of your own.

**Expected:** All eight old behaviours remain correct. Feature status remains NOT_RUN until you explicitly enable its check.


In [ ]:
after_checks = baseline_tests()
RUN_FEATURE_CHECK = False
feature_check = {"status": "NOT_RUN"}
if RUN_FEATURE_CHECK:
    try:
        Booking = load_booking_class()
        booking = Booking(2); booking.register("S001")
        original = list(booking.registered)
        actual = booking.status()
        expected = {"capacity":2, "registered_count":1, "remaining":1}
        feature_check = {"status":"PASS" if actual == expected and booking.registered == original else "FAIL",
                         "actual":actual, "expected":expected}
    except Exception as error:
        feature_check = {"status":"FAIL", "issue":type(error).__name__}
print("Feature check:", feature_check)
# Rebuild source references after any reviewed edit.
print("Current indexed blocks:", len(build_code_index()))


## 9 Save what you actually did

**What this cell does:** Records baseline results, feature status and the current source hash. It can export the three project files and evidence as a ZIP.

Fill in your actual route and explanation. A passing baseline does not imply you completed the new feature. Keep prompts and screenshots free of credentials.


In [ ]:
ASSISTANCE_ROUTE = "not recorded yet"  # Example: browser assistant; local coding client; manual implementation.
MY_EXPLANATION = ""  # Explain one line you changed and why the original tests matter.
evidence = {"mode": MODE, "retrieval_mode":"RETRIEVAL_ONLY", "baseline_before":before_checks,
            "baseline_after":after_checks, "feature_check":feature_check,
            "assistance_route":ASSISTANCE_ROUTE, "my_explanation":MY_EXPLANATION,
            "bookings_sha256":hashlib.sha256((WORKSPACE / "bookings.py").read_bytes()).hexdigest(),
            "model_requests_by_this_notebook":0}
(WORKSPACE / "evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print(json.dumps(evidence, indent=2))
DOWNLOAD_PROJECT = False
if DOWNLOAD_PROJECT:
    archive = WORKSPACE / "my_code_project.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for filename in ["bookings.py", "app.py", "README.md", "evidence.json"]:
            bundle.write(WORKSPACE / filename, filename)
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print("ZIP saved in the temporary workspace for your local notebook environment.")


## Show your neighbour

1. Show a source reference that supports your explanation.
2. Show the reviewed code diff and eight original checks.
3. Demonstrate status() and explain why its output matches the actual registrations.

**Further work:** Add a small local interface, or replace lexical retrieval with an embedding index after the workshop. Keep sources and evaluation checks when you change the retrieval method.

**Credit:** Project inspiration: patchy631, AI Engineering Hub, Chat with Code, pinned commit 042aeb5067384981eeba6a07fa882f436bb1660d. This notebook contains original lightweight workshop code; it does not reproduce or execute the upstream application.
